In [3]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

print("Imports completed successfully.")

Imports completed successfully.


In [4]:
def find_project_root() -> Path:
    """
    Find the thesis project root.

    It checks the current folder and its parents for folders such as
    Notebooks, Datasets, Processed, or Results.
    """
    current = Path.cwd().resolve()

    candidate_paths = [current, *current.parents]

    for candidate in candidate_paths:
        expected_folders = [
            candidate / "Notebooks",
            candidate / "Datasets",
        ]

        if all(folder.exists() for folder in expected_folders):
            return candidate

    # Fallback: if notebook is running from the Notebooks folder
    if current.name.lower() == "notebooks":
        return current.parent

    # Final fallback: current directory
    return current


PROJECT_ROOT = find_project_root()

DATA_ROOT = PROJECT_ROOT / "Datasets"
PROCESSED_ROOT = PROJECT_ROOT / "Processed"
RESULTS_ROOT = PROJECT_ROOT / "Results"
VALIDATION_ROOT = RESULTS_ROOT / "data_validation"
PREPROCESSOR_ROOT = PROJECT_ROOT / "Models" / "preprocessors"

for folder in [
    DATA_ROOT,
    PROCESSED_ROOT,
    RESULTS_ROOT,
    VALIDATION_ROOT,
    PREPROCESSOR_ROOT,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("PROCESSED_ROOT:", PROCESSED_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)

PROJECT_ROOT: C:\Users\souha\Downloads\Human Centered AutoML Thesis
DATA_ROOT: C:\Users\souha\Downloads\Human Centered AutoML Thesis\Datasets
PROCESSED_ROOT: C:\Users\souha\Downloads\Human Centered AutoML Thesis\Processed
RESULTS_ROOT: C:\Users\souha\Downloads\Human Centered AutoML Thesis\Results


In [5]:
def find_project_root() -> Path:
    """
    Find the thesis project root.

    It checks the current folder and its parents for folders such as
    Notebooks, Datasets, Processed, or Results.
    """
    current = Path.cwd().resolve()

    candidate_paths = [current, *current.parents]

    for candidate in candidate_paths:
        expected_folders = [
            candidate / "Notebooks",
            candidate / "Datasets",
        ]

        if all(folder.exists() for folder in expected_folders):
            return candidate

    # Fallback: if notebook is running from the Notebooks folder
    if current.name.lower() == "notebooks":
        return current.parent

    # Final fallback: current directory
    return current


PROJECT_ROOT = find_project_root()

DATA_ROOT = PROJECT_ROOT / "Datasets"
PROCESSED_ROOT = PROJECT_ROOT / "Processed"
RESULTS_ROOT = PROJECT_ROOT / "Results"
VALIDATION_ROOT = RESULTS_ROOT / "data_validation"
PREPROCESSOR_ROOT = PROJECT_ROOT / "Models" / "preprocessors"

for folder in [
    DATA_ROOT,
    PROCESSED_ROOT,
    RESULTS_ROOT,
    VALIDATION_ROOT,
    PREPROCESSOR_ROOT,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("PROCESSED_ROOT:", PROCESSED_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)

PROJECT_ROOT: C:\Users\souha\Downloads\Human Centered AutoML Thesis
DATA_ROOT: C:\Users\souha\Downloads\Human Centered AutoML Thesis\Datasets
PROCESSED_ROOT: C:\Users\souha\Downloads\Human Centered AutoML Thesis\Processed
RESULTS_ROOT: C:\Users\souha\Downloads\Human Centered AutoML Thesis\Results


In [6]:
SEEDS = [42, 123, 2026]
TEST_SIZE = 0.20
TARGET_COLUMN = "target"

print("Seeds:", SEEDS)
print("Test size:", TEST_SIZE)

Seeds: [42, 123, 2026]
Test size: 0.2


In [7]:
SEEDS = [42, 123, 2026]
TEST_SIZE = 0.20
TARGET_COLUMN = "target"

print("Seeds:", SEEDS)
print("Test size:", TEST_SIZE)

Seeds: [42, 123, 2026]
Test size: 0.2


In [8]:
def clean_column_name(column_name: str) -> str:
    """
    Convert column names to lowercase snake_case-like names.
    """
    cleaned = str(column_name).strip().lower()

    replacements = {
        " ": "_",
        ".": "_",
        "-": "_",
        "/": "_",
    }

    for old, new in replacements.items():
        cleaned = cleaned.replace(old, new)

    while "__" in cleaned:
        cleaned = cleaned.replace("__", "_")

    return cleaned.strip("_")


def clean_dataframe_columns(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Return a copy with standardized column names.
    """
    cleaned_dataframe = dataframe.copy()
    cleaned_dataframe.columns = [
        clean_column_name(column)
        for column in cleaned_dataframe.columns
    ]
    return cleaned_dataframe

In [9]:
def load_breast_cancer_dataset():
    """
    Load the Breast Cancer Wisconsin Diagnostic dataset.

    Returns
    -------
    X : pd.DataFrame
        Predictor variables.
    y : pd.Series
        Target variable.
    metadata : dict
        Dataset information.
    """
    dataset = load_breast_cancer(as_frame=True)

    X = dataset.data.copy()
    y = dataset.target.copy()

    X = clean_dataframe_columns(X)
    y = y.rename(TARGET_COLUMN)

    metadata = {
        "dataset": "breast_cancer",
        "source": "scikit-learn load_breast_cancer",
        "classification_type": "binary",
        "original_rows": int(X.shape[0]),
        "original_features": int(X.shape[1]),
        "numerical_features": int(X.shape[1]),
        "categorical_features": 0,
        "target_classes": int(y.nunique()),
    }

    return X, y, metadata


X_breast_cancer, y_breast_cancer, breast_cancer_metadata = (
    load_breast_cancer_dataset()
)

print("Breast Cancer X shape:", X_breast_cancer.shape)
print("Breast Cancer y shape:", y_breast_cancer.shape)
print("\nTarget distribution:")
print(y_breast_cancer.value_counts(normalize=True).sort_index())

Breast Cancer X shape: (569, 30)
Breast Cancer y shape: (569,)

Target distribution:
target
0    0.372583
1    0.627417
Name: proportion, dtype: float64


In [10]:
def load_wine_dataset():
    """
    Load the Wine Recognition dataset.
    """
    dataset = load_wine(as_frame=True)

    X = dataset.data.copy()
    y = dataset.target.copy()

    X = clean_dataframe_columns(X)
    y = y.rename(TARGET_COLUMN)

    metadata = {
        "dataset": "wine",
        "source": "scikit-learn load_wine",
        "classification_type": "multiclass",
        "original_rows": int(X.shape[0]),
        "original_features": int(X.shape[1]),
        "numerical_features": int(X.shape[1]),
        "categorical_features": 0,
        "target_classes": int(y.nunique()),
    }

    return X, y, metadata


X_wine, y_wine, wine_metadata = load_wine_dataset()

print("Wine X shape:", X_wine.shape)
print("Wine y shape:", y_wine.shape)
print("\nTarget distribution:")
print(y_wine.value_counts(normalize=True).sort_index())

Wine X shape: (178, 13)
Wine y shape: (178,)

Target distribution:
target
0    0.331461
1    0.398876
2    0.269663
Name: proportion, dtype: float64


In [11]:
def find_titanic_file() -> Path:
    """
    Search recursively inside Datasets for a likely Titanic CSV file.
    """
    possible_exact_names = {
        "titanic.csv",
        "titanic3.csv",
        "titanic_dataset.csv",
        "titanic_data.csv",
    }

    csv_files = list(DATA_ROOT.rglob("*.csv"))

    # First, search exact common filenames
    for file_path in csv_files:
        if file_path.name.lower() in possible_exact_names:
            return file_path

    # Then, search any CSV containing 'titanic'
    for file_path in csv_files:
        if "titanic" in file_path.name.lower():
            return file_path

    raise FileNotFoundError(
        "No Titanic CSV file was found inside the Datasets folder. "
        "Add titanic.csv or titanic3.csv to the Datasets folder."
    )


TITANIC_FILE = find_titanic_file()

print("Titanic file found:")
print(TITANIC_FILE)

Titanic file found:
C:\Users\souha\Downloads\Human Centered AutoML Thesis\Datasets\titanic.csv


In [12]:
titanic_raw = pd.read_csv(TITANIC_FILE)
titanic_raw = clean_dataframe_columns(titanic_raw)

print("Raw Titanic shape:", titanic_raw.shape)
print("\nAvailable columns:")
print(titanic_raw.columns.tolist())

display(titanic_raw.head())

Raw Titanic shape: (1309, 14)

Available columns:
['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked', 'boat', 'body', 'home_dest']


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home_dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


In [15]:
def load_titanic_dataset(file_path: Path):
    """
    Load Titanic and retain only the predictors defined in the thesis.

    Retained:
    - pclass
    - sex
    - age
    - sibsp
    - parch
    - fare
    - embarked

    Target:
    - survived

    Explicitly excluded:
    - boat and body because of target leakage
    - name, ticket, cabin, and home_dest because of high cardinality,
      incompleteness, or controlled-comparison considerations
    """
    dataframe = pd.read_csv(file_path)
    dataframe = clean_dataframe_columns(dataframe)

    # Support alternative spellings
    rename_map = {}

    if "home_dest" not in dataframe.columns:
        if "home_destination" in dataframe.columns:
            rename_map["home_destination"] = "home_dest"
        elif "homedest" in dataframe.columns:
            rename_map["homedest"] = "home_dest"

    if "siblings_spouses_aboard" in dataframe.columns:
        rename_map["siblings_spouses_aboard"] = "sibsp"

    if "parents_children_aboard" in dataframe.columns:
        rename_map["parents_children_aboard"] = "parch"

    dataframe = dataframe.rename(columns=rename_map)

    retained_features = [
        "pclass",
        "sex",
        "age",
        "sibsp",
        "parch",
        "fare",
        "embarked",
    ]

    target_candidate_names = [
        "survived",
        "target",
    ]

    target_name = None

    for candidate in target_candidate_names:
        if candidate in dataframe.columns:
            target_name = candidate
            break

    if target_name is None:
        raise KeyError(
            "The Titanic target column was not found. "
            "Expected 'survived' or 'target'."
        )

    missing_required_columns = [
        column
        for column in retained_features
        if column not in dataframe.columns
    ]

    if missing_required_columns:
        raise KeyError(
            "The following required Titanic columns are missing: "
            f"{missing_required_columns}. "
            f"Available columns: {dataframe.columns.tolist()}"
        )

    X = dataframe[retained_features].copy()
    y = dataframe[target_name].copy().rename(TARGET_COLUMN)

    # Convert known categorical columns to string/category-friendly values
    categorical_columns = ["sex", "embarked"]

    for column in categorical_columns:
        X[column] = X[column].astype("object")

    # Ensure target is numeric
    y = pd.to_numeric(y, errors="raise").astype(int)

    metadata = {
        "dataset": "titanic",
        "source": str(file_path),
        "classification_type": "binary",
        "original_rows": int(X.shape[0]),
        "original_features": int(X.shape[1]),
        "numerical_features": 5,
        "categorical_features": 2,
        "target_classes": int(y.nunique()),
        "retained_features": retained_features,
        "excluded_leakage_columns": ["boat", "body"],
        "other_excluded_columns": [
            "name",
            "ticket",
            "cabin",
            "home_dest",
        ],
    }

    return X, y, metadata


X_titanic, y_titanic, titanic_metadata = load_titanic_dataset(
    TITANIC_FILE
)

print("Titanic X shape:", X_titanic.shape)
print("Titanic y shape:", y_titanic.shape)

print("\nRetained features:")
print(X_titanic.columns.tolist())

print("\nMissing values:")
print(X_titanic.isna().sum())

print("\nTarget distribution:")
print(y_titanic.value_counts(normalize=True).sort_index())

Titanic X shape: (1309, 7)
Titanic y shape: (1309,)

Retained features:
['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']

Missing values:
pclass        0
sex           0
age         263
sibsp         0
parch         0
fare          1
embarked      2
dtype: int64

Target distribution:
target
0    0.618029
1    0.381971
Name: proportion, dtype: float64


In [16]:
DATASETS = {
    "breast_cancer": {
        "X": X_breast_cancer,
        "y": y_breast_cancer,
        "metadata": breast_cancer_metadata,
    },
    "wine": {
        "X": X_wine,
        "y": y_wine,
        "metadata": wine_metadata,
    },
    "titanic": {
        "X": X_titanic,
        "y": y_titanic,
        "metadata": titanic_metadata,
    },
}

for dataset_name, dataset_information in DATASETS.items():
    X = dataset_information["X"]
    y = dataset_information["y"]

    print(
        f"{dataset_name}: "
        f"X={X.shape}, "
        f"y={y.shape}, "
        f"classes={y.nunique()}"
    )

breast_cancer: X=(569, 30), y=(569,), classes=2
wine: X=(178, 13), y=(178,), classes=3
titanic: X=(1309, 7), y=(1309,), classes=2


In [17]:
def identify_column_types(X: pd.DataFrame):
    """
    Identify numerical and categorical columns.
    """
    categorical_columns = X.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numerical_columns = [
        column
        for column in X.columns
        if column not in categorical_columns
    ]

    return numerical_columns, categorical_columns


for dataset_name, dataset_information in DATASETS.items():
    X = dataset_information["X"]

    numerical_columns, categorical_columns = identify_column_types(X)

    print(f"\nDataset: {dataset_name}")
    print("Numerical columns:", numerical_columns)
    print("Categorical columns:", categorical_columns)


Dataset: breast_cancer
Numerical columns: ['mean_radius', 'mean_texture', 'mean_perimeter', 'mean_area', 'mean_smoothness', 'mean_compactness', 'mean_concavity', 'mean_concave_points', 'mean_symmetry', 'mean_fractal_dimension', 'radius_error', 'texture_error', 'perimeter_error', 'area_error', 'smoothness_error', 'compactness_error', 'concavity_error', 'concave_points_error', 'symmetry_error', 'fractal_dimension_error', 'worst_radius', 'worst_texture', 'worst_perimeter', 'worst_area', 'worst_smoothness', 'worst_compactness', 'worst_concavity', 'worst_concave_points', 'worst_symmetry', 'worst_fractal_dimension']
Categorical columns: []

Dataset: wine
Numerical columns: ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280_od315_of_diluted_wines', 'proline']
Categorical columns: []

Dataset: titanic
Numerical columns: ['pclass', 'age', 'sibsp', 'parch', 'fare']
Categori

In [18]:
def build_preprocessor(X: pd.DataFrame):
    """
    Build a preprocessing pipeline based on feature types.

    Numerical:
    - median imputation
    - standardization

    Categorical:
    - most-frequent imputation
    - one-hot encoding
    """
    numerical_columns, categorical_columns = identify_column_types(X)

    transformers = []

    if numerical_columns:
        numerical_pipeline = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
                (
                    "scaler",
                    StandardScaler(),
                ),
            ]
        )

        transformers.append(
            (
                "numeric",
                numerical_pipeline,
                numerical_columns,
            )
        )

    if categorical_columns:
        categorical_pipeline = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]
        )

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_columns,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )

    return preprocessor, numerical_columns, categorical_columns

In [19]:
def class_distribution_as_json(y: pd.Series) -> str:
    """
    Return target class proportions as a JSON string.
    """
    distribution = (
        y.value_counts(normalize=True)
        .sort_index()
        .round(6)
        .to_dict()
    )

    serializable_distribution = {
        str(key): float(value)
        for key, value in distribution.items()
    }

    return json.dumps(
        serializable_distribution,
        sort_keys=True,
    )

In [20]:
def count_conflicting_duplicates(
    X: pd.DataFrame,
    y: pd.Series,
) -> int:
    """
    Count feature rows that occur with more than one target label.
    """
    combined = X.reset_index(drop=True).copy()
    combined[TARGET_COLUMN] = y.reset_index(drop=True)

    target_counts_per_feature_row = (
        combined.groupby(list(X.columns), dropna=False)[TARGET_COLUMN]
        .nunique()
    )

    return int((target_counts_per_feature_row > 1).sum())

In [21]:
def prepare_validate_and_save_dataset(
    dataset_name: str,
    X: pd.DataFrame,
    y: pd.Series,
    metadata: dict,
    seed: int,
) -> dict:
    """
    Split, preprocess, validate, and save one dataset for one seed.
    """
    # Ensure clean positional alignment
    X = X.reset_index(drop=True).copy()
    y = y.reset_index(drop=True).copy()

    if len(X) != len(y):
        raise ValueError(
            f"{dataset_name}: X and y have different lengths."
        )

    # Preserve original row IDs before splitting
    row_ids = pd.Series(
        np.arange(len(X)),
        index=X.index,
        name="original_row_id",
    )

    (
        X_train_raw,
        X_test_raw,
        y_train,
        y_test,
        train_row_ids,
        test_row_ids,
    ) = train_test_split(
        X,
        y,
        row_ids,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=seed,
    )

    # Create preprocessing pipeline using training feature structure
    (
        preprocessor,
        numerical_columns,
        categorical_columns,
    ) = build_preprocessor(X_train_raw)

    # IMPORTANT:
    # Fit only on training data
    X_train_array = preprocessor.fit_transform(X_train_raw)

    # Apply already fitted transformation to test data
    X_test_array = preprocessor.transform(X_test_raw)

    feature_names = preprocessor.get_feature_names_out().tolist()

    X_train_processed = pd.DataFrame(
        X_train_array,
        columns=feature_names,
    )

    X_test_processed = pd.DataFrame(
        X_test_array,
        columns=feature_names,
    )

    y_train_saved = y_train.reset_index(drop=True).rename(
        TARGET_COLUMN
    )
    y_test_saved = y_test.reset_index(drop=True).rename(
        TARGET_COLUMN
    )

    train_row_ids_saved = train_row_ids.reset_index(
        drop=True
    ).astype(int)

    test_row_ids_saved = test_row_ids.reset_index(
        drop=True
    ).astype(int)

    output_directory = (
        PROCESSED_ROOT
        / dataset_name
        / f"seed_{seed}"
    )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    preprocessor_directory = (
        PREPROCESSOR_ROOT
        / dataset_name
        / f"seed_{seed}"
    )

    preprocessor_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Save processed data
    X_train_processed.to_csv(
        output_directory / "X_train.csv",
        index=False,
    )

    X_test_processed.to_csv(
        output_directory / "X_test.csv",
        index=False,
    )

    y_train_saved.to_frame().to_csv(
        output_directory / "y_train.csv",
        index=False,
    )

    y_test_saved.to_frame().to_csv(
        output_directory / "y_test.csv",
        index=False,
    )

    # Save raw split before preprocessing
    X_train_raw.reset_index(drop=True).to_csv(
        output_directory / "X_train_raw.csv",
        index=False,
    )

    X_test_raw.reset_index(drop=True).to_csv(
        output_directory / "X_test_raw.csv",
        index=False,
    )

    # Save original row IDs
    pd.DataFrame(
        {
            "original_row_id": train_row_ids_saved,
        }
    ).to_csv(
        output_directory / "train_row_ids.csv",
        index=False,
    )

    pd.DataFrame(
        {
            "original_row_id": test_row_ids_saved,
        }
    ).to_csv(
        output_directory / "test_row_ids.csv",
        index=False,
    )

    # Save fitted preprocessor
    preprocessor_path = (
        preprocessor_directory
        / "preprocessor.joblib"
    )

    joblib.dump(
        preprocessor,
        preprocessor_path,
    )

    # Save feature names
    pd.DataFrame(
        {
            "processed_feature": feature_names,
        }
    ).to_csv(
        output_directory / "processed_feature_names.csv",
        index=False,
    )

    # Validation checks
    train_test_columns_equal = (
        X_train_processed.columns.tolist()
        == X_test_processed.columns.tolist()
    )

    train_missing_values = int(
        X_train_processed.isna().sum().sum()
    )

    test_missing_values = int(
        X_test_processed.isna().sum().sum()
    )

    train_test_row_overlap = len(
        set(train_row_ids_saved)
        .intersection(set(test_row_ids_saved))
    )

    complete_row_coverage = (
        len(
            set(train_row_ids_saved)
            .union(set(test_row_ids_saved))
        )
        == len(X)
    )

    duplicate_raw_feature_rows = int(
        X.duplicated().sum()
    )

    duplicate_train_processed_rows = int(
        X_train_processed.duplicated().sum()
    )

    duplicate_test_processed_rows = int(
        X_test_processed.duplicated().sum()
    )

    conflicting_duplicate_rows = (
        count_conflicting_duplicates(X, y)
    )

    validation_status = "PASS"

    failure_reasons = []

    if train_missing_values != 0:
        failure_reasons.append(
            "Missing values remain in training data."
        )

    if test_missing_values != 0:
        failure_reasons.append(
            "Missing values remain in test data."
        )

    if not train_test_columns_equal:
        failure_reasons.append(
            "Training and test columns differ."
        )

    if train_test_row_overlap != 0:
        failure_reasons.append(
            "Training and test row IDs overlap."
        )

    if not complete_row_coverage:
        failure_reasons.append(
            "The split does not cover every original row exactly once."
        )

    if X_train_processed.shape[0] != len(y_train_saved):
        failure_reasons.append(
            "Training feature and target lengths differ."
        )

    if X_test_processed.shape[0] != len(y_test_saved):
        failure_reasons.append(
            "Test feature and target lengths differ."
        )

    if failure_reasons:
        validation_status = "FAIL"

    validation_row = {
        "dataset": dataset_name,
        "seed": seed,
        "source": metadata.get("source"),
        "original_rows": int(X.shape[0]),
        "original_features": int(X.shape[1]),
        "raw_numerical_features": len(numerical_columns),
        "raw_categorical_features": len(categorical_columns),
        "train_rows": int(X_train_processed.shape[0]),
        "test_rows": int(X_test_processed.shape[0]),
        "processed_features": int(
            X_train_processed.shape[1]
        ),
        "target_classes": int(y.nunique()),
        "train_missing_values": train_missing_values,
        "test_missing_values": test_missing_values,
        "train_class_distribution": (
            class_distribution_as_json(y_train_saved)
        ),
        "test_class_distribution": (
            class_distribution_as_json(y_test_saved)
        ),
        "train_test_columns_equal": (
            train_test_columns_equal
        ),
        "train_test_row_overlap": (
            train_test_row_overlap
        ),
        "complete_row_coverage": (
            complete_row_coverage
        ),
        "duplicate_raw_feature_rows": (
            duplicate_raw_feature_rows
        ),
        "duplicate_train_processed_rows": (
            duplicate_train_processed_rows
        ),
        "duplicate_test_processed_rows": (
            duplicate_test_processed_rows
        ),
        "conflicting_duplicate_feature_rows": (
            conflicting_duplicate_rows
        ),
        "preprocessing_fitted_on_training_only": True,
        "output_directory": str(output_directory),
        "preprocessor_path": str(preprocessor_path),
        "status": validation_status,
        "failure_reasons": " | ".join(failure_reasons),
    }

    return validation_row

In [22]:
validation_rows = []

for dataset_name, dataset_information in DATASETS.items():
    X = dataset_information["X"]
    y = dataset_information["y"]
    metadata = dataset_information["metadata"]

    print("\n" + "=" * 80)
    print(f"PROCESSING DATASET: {dataset_name}")
    print("=" * 80)

    for seed in SEEDS:
        print(f"\nProcessing seed {seed}...")

        validation_row = prepare_validate_and_save_dataset(
            dataset_name=dataset_name,
            X=X,
            y=y,
            metadata=metadata,
            seed=seed,
        )

        validation_rows.append(validation_row)

        print(
            f"Status: {validation_row['status']} | "
            f"Train rows: {validation_row['train_rows']} | "
            f"Test rows: {validation_row['test_rows']} | "
            f"Processed features: "
            f"{validation_row['processed_features']}"
        )

print("\nAll dataset-seed combinations have been processed.")


PROCESSING DATASET: breast_cancer

Processing seed 42...
Status: PASS | Train rows: 455 | Test rows: 114 | Processed features: 30

Processing seed 123...
Status: PASS | Train rows: 455 | Test rows: 114 | Processed features: 30

Processing seed 2026...
Status: PASS | Train rows: 455 | Test rows: 114 | Processed features: 30

PROCESSING DATASET: wine

Processing seed 42...
Status: PASS | Train rows: 142 | Test rows: 36 | Processed features: 13

Processing seed 123...
Status: PASS | Train rows: 142 | Test rows: 36 | Processed features: 13

Processing seed 2026...
Status: PASS | Train rows: 142 | Test rows: 36 | Processed features: 13

PROCESSING DATASET: titanic

Processing seed 42...
Status: PASS | Train rows: 1047 | Test rows: 262 | Processed features: 10

Processing seed 123...
Status: PASS | Train rows: 1047 | Test rows: 262 | Processed features: 10

Processing seed 2026...
Status: PASS | Train rows: 1047 | Test rows: 262 | Processed features: 10

All dataset-seed combinations have b

In [25]:
validation_report = pd.DataFrame(validation_rows)

validation_report = validation_report.sort_values(
    by=["dataset", "seed"]
).reset_index(drop=True)

display(
    validation_report[
        [
            "dataset",
            "seed",
            "original_rows",
            "original_features",
            "train_rows",
            "test_rows",
            "processed_features",
            "train_missing_values",
            "test_missing_values",
            "train_test_columns_equal",
            "train_test_row_overlap",
            "complete_row_coverage",
            "status",
        ]
    ]
)

,dataset,seed,original_rows,original_features,train_rows,test_rows,processed_features,train_missing_values,test_missing_values,train_test_columns_equal,train_test_row_overlap,complete_row_coverage,status
0,breast_cancer,42,569,30,455,114,30,0,0,True,0,True,PASS
1,breast_cancer,123,569,30,455,114,30,0,0,True,0,True,PASS
2,breast_cancer,2026,569,30,455,114,30,0,0,True,0,True,PASS
3,titanic,42,1309,7,1047,262,10,0,0,True,0,True,PASS
4,titanic,123,1309,7,1047,262,10,0,0,True,0,True,PASS
5,titanic,2026,1309,7,1047,262,10,0,0,True,0,True,PASS
6,wine,42,178,13,142,36,13,0,0,True,0,True,PASS
7,wine,123,178,13,142,36,13,0,0,True,0,True,PASS
8,wine,2026,178,13,142,36,13,0,0,True,0,True,PASS


In [27]:
%pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [28]:
validation_report_path = (
    VALIDATION_ROOT
    / "data_validation_report.csv"
)

validation_report.to_csv(
    validation_report_path,
    index=False,
)

validation_report_excel_path = (
    VALIDATION_ROOT
    / "data_validation_report.xlsx"
)

validation_report.to_excel(
    validation_report_excel_path,
    index=False,
)

print("CSV report saved to:")
print(validation_report_path)

print("\nExcel report saved to:")
print(validation_report_excel_path)

CSV report saved to:
C:\Users\souha\Downloads\Human Centered AutoML Thesis\Results\data_validation\data_validation_report.csv

Excel report saved to:
C:\Users\souha\Downloads\Human Centered AutoML Thesis\Results\data_validation\data_validation_report.xlsx
